In [ ]:

import warnings; warnings.filterwarnings("ignore")
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
!pip install pennylane pennylane-lightning --upgrade -q

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__} CUDA: {torch.cuda.is_available()}")

In [ ]:
INPUT_DIR = "/kaggle/input/datasets/parthenaik/q-sentinel/Qsentinel"
OUTPUT_DIR = Path("/kaggle/working"); OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
scripts_dir = "/kaggle/input/datasets/parthenaik/my-cicio-scripts"
if scripts_dir not in sys.path: sys.path.append(scripts_dir)

import scripts

print("scripts loaded from:", scripts.__file__)

In [ ]:


from scripts.cache import (
    ForwardCache,
    cached_calibrate_threshold,
    cached_conformal_alpha_sweep,
    cached_lipschitz_percentile,
    cached_qsnet_infer,
)
from scripts.circuit import build_forward_circuit, create_quantum_device
from scripts.constants import (
    BARREN_PLATEAU_VAR_THRESHOLD,
    DEFAULT_ALPHA,
    DEFAULT_BATCH_SIZE,
    DEFAULT_CF,
    DEFAULT_NOISE_RATE,
    DEFAULT_REUPLOAD,
    ZERO_DAY,
)
from scripts.data import (
    greedy_dpp_sample,
    load_split,
    to_angles,
)
from scripts.gradient import gradient_variance_probe
from scripts.hilbert import hilbert_geometry_diagnostics, print_h1_report
from scripts.inference import predict_labels
from scripts.logging import write_history_log
from scripts.memory import is_oom_error, run_batched_safely, safe_empty_cache
from scripts.prototypes import prototype_summary
from scripts.train import train_maqt
from scripts.utils import get_torch_device, to_np_y

print("imports OK")

In [ ]:

NOTEBOOK_NAME = "maqt-hptuned-cicio"
dataset = "CICIoT2023"
target_col = "label_multiclass"
DATA_DIR = "/kaggle/input/datasets/parthenaik/q-sentinel-cicio/Qsentinel/CICIoT2023"
FROZEN_JSON_PATH = "/kaggle/input/datasets/parthenaik/q-sentinel-cicio/Qsentinel/scripts/teamC_week3_FROZEN_handoff.json"

OUT_DIR = OUTPUT_DIR / "models" / "prototypes" / dataset; OUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR = OUTPUT_DIR / "logs"; LOG_DIR.mkdir(parents=True, exist_ok=True)

handoff = json.loads(Path(FROZEN_JSON_PATH).read_text())
FEATURE_COLS = list(handoff["frozen_subsets"][dataset]["features"])
print(f"selector={handoff['frozen_subsets'][dataset]['selector']} k={len(FEATURE_COLS)}")

X_train_full, y_train_full, class_names, df_train = load_split(
    DATA_DIR, "train", target_col, csv=True, selected_cols=FEATURE_COLS, return_df=True)
X_cal_full, y_cal_full, _, df_cal = load_split(
    DATA_DIR, "calibration", target_col, csv=True, selected_cols=FEATURE_COLS, return_df=True)
X_test_full, y_test_full, _, df_test = load_split(
    DATA_DIR, "test", target_col, csv=True, selected_cols=FEATURE_COLS, return_df=True)
X_zero_full, y_zero_full, _, df_zero = load_split(
    DATA_DIR, "zeroday", target_col, csv=True, selected_cols=FEATURE_COLS, return_df=True)

print(f"train {X_train_full.shape} cal {X_cal_full.shape} test {X_test_full.shape} zero {X_zero_full.shape}")
print(f"classes: {class_names}")
num_classes = len(class_names)


In [ ]:
df_train.label_multiclass.value_counts()

In [ ]:

scaler = StandardScaler()
X_train_full_scaled = scaler.fit_transform(X_train_full)

USE_PCA, TARGET_VARIANCE = True, 0.95
pca_full = PCA(random_state=SEED).fit(X_train_full_scaled)
cum = np.cumsum(pca_full.explained_variance_ratio_)
k95 = min(int(np.searchsorted(cum, TARGET_VARIANCE) + 1), len(cum))
print(f"k95 components: {k95}")

pca = PCA(n_components=k95, random_state=SEED)
X_train_full_feat = pca.fit_transform(X_train_full_scaled)
x_min, x_max = X_train_full_feat.min(0), X_train_full_feat.max(0)

SUBSET, PER_CLASS_CAP = True, 1000
if SUBSET:
    X_train, y_train = greedy_dpp_sample(X_train_full, y_train_full, per_class_cap=PER_CLASS_CAP, seed=SEED)
else:
    X_train, y_train = X_train_full, y_train_full
print(f"train coreset: {X_train.shape}")

def _to_ang(X): return to_angles(X, scaler, x_min, x_max, pca=pca, angle_max=np.pi)

X_train_angles = _to_ang(X_train)
X_cal_angles   = _to_ang(X_cal_full)
X_test_angles  = _to_ang(X_test_full)
X_zero_angles  = _to_ang(X_zero_full)
print("angle shapes:", X_train_angles.shape, X_cal_angles.shape, X_test_angles.shape, X_zero_angles.shape)

In [ ]:

num_qubits = int(k95)
num_layers = 3
noise_rate = DEFAULT_NOISE_RATE
REUPLOAD = DEFAULT_REUPLOAD

device = get_torch_device()
dev = create_quantum_device(num_qubits)
forward_circuit = build_forward_circuit(dev, num_qubits, num_layers, noise_rate=noise_rate, reupload=REUPLOAD)
print(f"device={device} wires={num_qubits}")

safe_empty_cache()
try:
    grad_var, _ = gradient_variance_probe(
        forward_circuit, X_train_angles[:1], num_qubits, num_layers,
        n_trials=5, device=torch.device("cpu"),
    )
    print(f"pre-train grad var = {grad_var:.2e} (thr={BARREN_PLATEAU_VAR_THRESHOLD:.0e})")
except RuntimeError as e:
    if is_oom_error(e): safe_empty_cache()
    else: raise

In [ ]:

epochs = 15
batch_size = 8
lambda1 = 0.5
lambda2 = 0.66
lr = 0.005
early_stopping = True
patience = 5
min_delta = 0.001

ckpt_dir = OUT_DIR / f"{NOTEBOOK_NAME}_seed{SEED}"
ckpt_dir.mkdir(parents=True, exist_ok=True)


resume_from = Path("/kaggle/input/models/parthenaik/cicio3-retraining-model/other/default/1/maqt-latest.pt")
if not resume_from.exists():
    resume_from = ckpt_dir / "maqt-latest.pt"   # fallback
assert resume_from.exists(), f"checkpoint not found: {resume_from}"
print(f"resuming from: {resume_from}")

torch.manual_seed(SEED)
result = run_batched_safely(
    train_maqt, X_train_angles, y_train,
    n_classes=num_classes, n_qubits=num_qubits, n_layers=num_layers,
    forward_circuit=forward_circuit, device=device,
    epochs=epochs, lr=lr, batch_size=batch_size, min_batch=2,
    label="train_maqt",
    lambda1_max=lambda1, lambda2_max=lambda2,
    use_weighted_sampler=True, seed=SEED,
    early_stopping=early_stopping, patience=patience, min_delta=min_delta,
    checkpoint_dir=ckpt_dir, save_every_epoch=True,
    resume_from=resume_from,
    reset_early_stopping_on_resume=True,     # fresh patience window for this leg
    heartbeat_every_steps=50, notebook_name=NOTEBOOK_NAME, log_dir=ckpt_dir,
)
if result is None:
    raise RuntimeError("train_maqt failed even at min_batch (OOM)")

theta_star, classifier_head, prototypes, ema_protos, history = result
safe_empty_cache()
print(f"epochs_ran={history.get('epochs_ran')} best_epoch={history.get('best_epoch')} "
      f"stop_reason={history.get('stop_reason')}")

In [ ]:

ckpt_path = OUT_DIR / f"{NOTEBOOK_NAME}-checkpoint.pt"
torch.save({
    "theta": theta_star.detach().cpu(),
    "head_state_dict": classifier_head.state_dict(),
    "prototypes": {k: v.detach().cpu() for k, v in prototypes.items()},
    "class_names": list(class_names),
    "num_classes": num_classes, "num_qubits": num_qubits, "num_layers": num_layers,
    "noise_rate": noise_rate, "reupload": REUPLOAD, "feature_cols": list(FEATURE_COLS),
    "lambda1": lambda1, "lambda2": lambda2, "lr": lr,
    "scaler": scaler, "pca": pca, "k95": int(k95),
    "angle_x_min": np.asarray(x_min), "angle_x_max": np.asarray(x_max),
    "angle_min": 0.0, "angle_max": float(np.pi),
}, ckpt_path)
print("published:", ckpt_path.resolve())

proto_df = pd.DataFrame(prototype_summary(prototypes, class_names))
proto_df

In [ ]:

y_true, y_pred = predict_labels(X_test_angles, y_test_full, theta_star, classifier_head,
                                 forward_circuit, device=device, batch_size=DEFAULT_BATCH_SIZE)
acc = accuracy_score(y_true, y_pred)
print(f"test accuracy: {acc:.4f}\n")
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=range(num_classes))
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(num_classes)); ax.set_xticklabels(class_names, rotation=45, ha="right")
ax.set_yticks(range(num_classes)); ax.set_yticklabels(class_names)
for i in range(num_classes):
    for j in range(num_classes):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max()/2 else "black")
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title("Confusion matrix")
plt.colorbar(im); plt.tight_layout(); plt.show()



In [ ]:

h1_report = hilbert_geometry_diagnostics(
    theta_star, X_test_angles, y_test_full, prototypes, forward_circuit,
    class_names=class_names, device=device, max_per_class=2000, seed=SEED,
)
print_h1_report(h1_report)



In [ ]:

cache = ForwardCache(store_device="cpu")
entry_cal   = cache.compute("cal",   X_cal_angles,  theta_star, forward_circuit, device=device, batch_size=64)
entry_test  = cache.compute("test",  X_test_angles, theta_star, forward_circuit, device=device, batch_size=64)
entry_zero  = cache.compute("zero",  X_zero_angles, theta_star, forward_circuit, device=device, batch_size=64)


q, scores_sorted = cached_calibrate_threshold(entry_cal, prototypes, alpha=DEFAULT_ALPHA)
print(f"conformal threshold q = {q:.4f} (alpha={DEFAULT_ALPHA}, n_cal={len(scores_sorted)})")


lip = cached_lipschitz_percentile(entry_test, n_pairs=300, percentile=95)
L_phi = lip["p95"]
print(f"L_phi (p95) = {L_phi:.4f}  max_ratio={lip['max']:.4f}")


labels_test, radii_test, scores_test, _ = cached_qsnet_infer(
    entry_test, prototypes, q, p=noise_rate, L_phi=L_phi, Cf=DEFAULT_CF, device=device,
)
known_mask = labels_test != ZERO_DAY
known_acc = float(np.mean(labels_test[known_mask] == to_np_y(y_test_full).astype(int)[known_mask])) if known_mask.any() else float("nan")
reject_rate_known = float(np.mean(~known_mask))
print(f"known-test accuracy (non-rejected): {known_acc:.4f}")
print(f"false-alarm (known flagged as zero-day): {reject_rate_known:.4f}")


labels_zero, radii_zero, scores_zero, _ = cached_qsnet_infer(
    entry_zero, prototypes, q, p=noise_rate, L_phi=L_phi, Cf=DEFAULT_CF, device=device,
)
zero_day_detection_rate = float(np.mean(labels_zero == ZERO_DAY))
print(f"zero-day detection rate: {zero_day_detection_rate:.4f}  (n={len(labels_zero)})")


alpha_sweep_rows = cached_conformal_alpha_sweep(entry_cal, entry_test, prototypes,
                                                 alphas=(0.01, 0.05, 0.1, 0.2))
pd.DataFrame(alpha_sweep_rows)

In [ ]:

log_path = write_history_log(
    history, NOTEBOOK_NAME,
    extra={
        "best_lambda2": BEST_LAMBDA2, "best_lr": BEST_LR,
        "sweep_grid": {"lambda2": LAMBDA2_GRID, "lr": LR_GRID},
        "test_accuracy": acc, "ddos_dos_confusion": int(ddos_dos_confusion),
        "h1_fidelity_gap": h1_report["fidelity_gap"],
        "conformal_q": float(q), "L_phi_p95": float(L_phi),
        "known_test_accuracy_nonrejected": known_acc,
        "zero_day_detection_rate": zero_day_detection_rate,
        "class_names": list(class_names), "seed": SEED,
    },
    log_dir=str(LOG_DIR),
)
print(f"wrote {Path(log_path).resolve()}")